In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs

def final_calibration_run(u_c, k_best, n_steps=10**8, n_bins=5000):
    # 使用 10^8 步进行终极校准
    x, gamma, n_offset = 0.5, 2, 1000000
    orbit = []
    
    print(f"--- 启动非自治校准模拟 (k={k_best}, steps={n_steps}) ---")
    
    # 模拟老化过程
    for n in range(2, n_offset + n_steps):
        u_n = u_c - k_best * (np.log(n))**(-gamma)
        x = 1 - max(u_n, 0.1) * x**2
        x = np.clip(x, -1.0, 1.0)
        if n >= n_offset: orbit.append(x)
    
    # 构造高分辨率转移矩阵
    orbit = np.array(orbit)
    bins = ((orbit + 1.0) / 2.0 * (n_bins - 1)).astype(int)
    P = csr_matrix((np.ones(len(bins)-1), (bins[:-1], bins[1:])), shape=(n_bins, n_bins))
    
    row_sums = np.array(P.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1.0
    P = P.multiply(1.0 / row_sums[:, np.newaxis])
    
    # 提取特征值相位
    vals, _ = eigs(P, k=100, which='LM', tol=1e-5)
    theta = np.sort(np.unique(np.round(np.abs(np.angle(vals[np.abs(vals) > 0.6])), 5)))
    theta = theta[theta > 0.05]
    
    # 对齐第一个零点并输出
    TRUE_GAMMAS = [14.1347, 21.0220, 25.0108, 30.4248, 32.9350]
    scaling = TRUE_GAMMAS[0] / theta[0]
    sim_gammas = theta[:5] * scaling
    
    print("\n校准后的前 5 模式位置：")
    for i, (s, t) in enumerate(zip(sim_gammas, TRUE_GAMMAS)):
        print(f"模式 {i+1}: 模拟={s:.4f} | 真实={t:.4f} | 误差={s-t:.4f}")

# 执行终极校准
u_c_fixed = 1.543689012
k_opt = 12.32
final_calibration_run(u_c_fixed, k_opt)

--- 启动非自治校准模拟 (k=12.32, steps=100000000) ---

校准后的前 5 模式位置：
模式 1: 模拟=14.1347 | 真实=14.1347 | 误差=0.0000
模式 2: 模拟=68.2111 | 真实=21.0220 | 误差=47.1891
模式 3: 模拟=74.0149 | 真实=25.0108 | 误差=49.0041
模式 4: 模拟=104.3711 | 真实=30.4248 | 误差=73.9463
模式 5: 模拟=131.1543 | 真实=32.9350 | 误差=98.2193
